# Tahap 3: Text Preprocessing

Notebook ini merepresentasikan blok abu-abu **Text Preprocessing** pada diagram metodologi penelitian, yang terdiri dari 5 sub-tahapan berurutan:
1. **Case Folding** (Mengubah seluruh huruf menjadi huruf kecil/*lowercase*)
2. **Tokenizing** (Memecah kalimat menjadi daftar kata/token)
3. **Normalization** (Mengubah kata tidak baku/slang/singkatan menjadi kata baku Bahasa Indonesia)
4. **Stopword Removal** (Menghapus kata umum yang tidak memiliki arti topik spesifik)
5. **Stemming** (Mengubah kata berimbuhan menjadi kata dasar menggunakan Sastrawi)

In [ ]:
import os
import re
import pandas as pd
import numpy as np
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

pd.set_option('display.max_colwidth', 180)

## 3.0 Memuat Data Hasil *Data Cleaning* (`02_cleaned_data.csv`)

In [ ]:
df = pd.read_csv('02_cleaned_data.csv')
print(f'Jumlah dokumen yang akan dipreprocessing: {len(df)}')
df.head(5)

## 3.1 Tahap 1: Case Folding
Mengubah semua karakter alfabet pada `text_cleaned` menjadi huruf kecil (*lowercase*) serta menormalisasi huruf berulang di akhir kata (seperti `terharuu` -> `terharu`, `jugaa` -> `juga`).

In [ ]:
def case_folding(text):
    text = str(text).lower()
    # Mengurangi huruf berulang >= 3 kali berurutan (contoh: gimanaaaa -> gimana)
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    return text

df['step1_casefolding'] = df['text_cleaned'].apply(case_folding)
df[['text_cleaned', 'step1_casefolding']].head(5)

## 3.2 Tahap 2: Tokenizing
Memecah teks hasil *Case Folding* menjadi potongan token kata tunggal dan membuang token yang hanya terdiri dari 1 karakter.

In [ ]:
def tokenize_text(text):
    tokens = re.findall(r'\b[a-z]{2,}\b', str(text))
    return tokens

df['step2_tokens'] = df['step1_casefolding'].apply(tokenize_text)
df[['step1_casefolding', 'step2_tokens']].head(5)

## 3.3 Tahap 3: Normalization (Kamus Kata Tidak Baku / Slang)
Menstandarkan kata tidak baku, singkatan media sosial X/Twitter, dan ejaan tidak standar pada korpus Karhutla ke dalam Bahasa Indonesia baku.

In [ ]:
slang_dict = {
    'yg': 'yang', 'ga': 'tidak', 'gak': 'tidak', 'gk': 'tidak', 'nggak': 'tidak',
    'tdk': 'tidak', 'tak': 'tidak', 'kagak': 'tidak', 'blm': 'belum', 'udh': 'sudah',
    'udah': 'sudah', 'sdh': 'sudah', 'dgn': 'dengan', 'dg': 'dengan', 'krn': 'karena',
    'karna': 'karena', 'utk': 'untuk', 'bgt': 'banget', 'bgtt': 'banget', 'dr': 'dari',
    'dlm': 'dalam', 'pd': 'pada', 'tp': 'tapi', 'tpi': 'tapi', 'kl': 'kalau',
    'klo': 'kalau', 'kalo': 'kalau', 'spt': 'seperti', 'kek': 'seperti', 'kayak': 'seperti',
    'kyk': 'seperti', 'bkn': 'bukan', 'jd': 'jadi', 'jdi': 'jadi', 'lg': 'lagi',
    'lgi': 'lagi', 'sm': 'sama', 'sma': 'sama', 'aja': 'saja', 'aj': 'saja',
    'org': 'orang', 'orng': 'orang', 'hrs': 'harus', 'bs': 'bisa', 'bsa': 'bisa',
    'jg': 'juga', 'jga': 'juga', 'jugaa': 'juga', 'allhamdulillah': 'alhamdulillah',
    'alhamdulilah': 'alhamdulillah', 'terharuu': 'terharu', 'nangiss': 'menangis',
    'nangisin': 'menangis', 'liat': 'lihat', 'ngeliat': 'melihat', 'ngelakuin': 'melakukan',
    'kurleb': 'kurang lebih', 'samsek': 'sama sekali', 'gue': 'saya', 'gw': 'saya',
    'aku': 'saya', 'lu': 'anda', 'lo': 'anda', 'u': 'anda', 'kalian': 'anda',
    ' indo ': 'indonesia', 'indo': 'indonesia', 'ri': 'republik indonesia',
    'kemlu': 'kementerian luar negeri', 'kalteng': 'kalimantan tengah',
    'kalbar': 'kalimantan barat', 'kalsel': 'kalimantan selatan',
    'kaltim': 'kalimantan timur', 'sumsel': 'sumatera selatan',
    'sumbar': 'sumatera barat', 'sumut': 'sumatera utara',
    'sumatra': 'sumatera', 'gub': 'gubernur', 'pemda': 'pemerintah daerah',
    'pemprov': 'pemerintah provinsi', 'menhut': 'menteri kehutanan',
    'menhan': 'menteri pertahanan', 'damkar': 'pemadam kebakaran',
    'bnpb': 'badan nasional penanggulangan bencana', 'bmkg': 'badan meteorologi klimatologi geofisika',
    'ispa': 'infeksi saluran pernapasan akut', 'tbtb': 'tiba tiba',
    'gegara': 'gara gara', 'ancur': 'hancur', 'kejebak': 'terjebak',
    'gimana': 'bagaimana', 'gmn': 'bagaimana', 'knp': 'kenapa', 'tau': 'tahu'
}

def normalize_tokens(tokens):
    normalized = []
    for token in tokens:
        replacement = slang_dict.get(token, token)
        # Jika hasil normalisasi terdiri dari >1 kata, pecah kembali menjadi token
        normalized.extend(replacement.split())
    return normalized

df['step3_normalized'] = df['step2_tokens'].apply(normalize_tokens)
df[['step2_tokens', 'step3_normalized']].head(5)

## 3.4 Tahap 4: Stopword Removal
Menghapus kata-kata umum (*stopwords*) Bahasa Indonesia dari Sastrawi ditambah daftar kata seruan/partikel media sosial yang tidak membawa informasi topik.

In [ ]:
stop_factory = StopWordRemoverFactory()
sastrawi_stopwords = set(stop_factory.get_stop_words())

# Tambahan custom stopwords untuk media sosial dan kata umum non-topik
custom_stopwords = {
    'yg', 'nya', 'ya', 'yaa', 'yah', 'sih', 'kok', 'kan', 'dong', 'deh', 'nih', 'tuh',
    'lah', 'kah', 'pun', 'kek', 'gitu', 'gini', 'begitu', 'begini', 'banget', ' banget',
    'guys', 'ges', 'gaes', 'bro', 'sis', 'pak', 'bu', 'bapak', 'ibu', 'kak', 'bang',
    'baca', 'klik', 'link', 'video', 'foto', 'berita', 'info', 'infonya', 'terkini',
    'hari', 'ini', 'itu', 'sana', 'sini', 'situ', 'udah', 'sudah', 'belum', 'lagi',
    'mau', 'akan', 'bisa', 'jadi', 'buat', 'bikin', 'sama', 'lebih', 'paling', 'sangat',
    'banyak', 'semua', 'seluruh', 'para', 'tiap', 'setiap', 'kalau', 'kalo', 'bila',
    'terus', 'lalu', 'kemudian', 'padahal', 'malah', 'justru', 'cuma', 'hanya', 'saja',
    'juga', 'pula', 'pun', 'dari', 'pada', 'ke', 'di', 'dan', 'atau', 'serta', 'dengan',
    'untuk', 'bagi', 'buat', 'karena', 'sebab', 'akibat', 'oleh', 'tentang', 'terhadap',
    'saya', 'aku', 'kami', 'kita', 'anda', 'kamu', 'mereka', 'dia', 'ia', 'beliau',
    'apa', 'siapa', 'mana', 'kapan', 'bagaimana', 'kenapa', 'mengapa', 'berapa',
    'tidak', 'bukan', 'jangan', 'tiada', 'tanpa', 'ada', 'adalah', 'ialah', 'merupakan',
    'yaitu', 'yakni', 'seperti', 'bagai', 'bak', 'laksana', 'ketika', 'saat', 'waktu',
    'masih', 'sedang', 'telah', 'pernah', 'sempat', 'harus', 'wajib', 'perlu', 'butuh',
    'ingin', 'hendak', 'bakal', 'coba', 'lihat', 'bilang', 'kata', 'katanya', 'sebut',
    'tahu', 'rasa', 'asa', 'punya', 'milik', 'dapat', 'kasih', 'beri', 'ambil', 'buat',
    'the', 'and', 'in', 'of', 'to', 'for', 'is', 'on', 'that', 'this', 'with', 'from',
    'bejir', 'bejirr', 'anjir', 'bjir', 'wkwk', 'wkwkwk', 'haha', 'hehe', 'huhu',
    'duar', 'lho', 'loh', 'noh', 'mah', 'sok', 'iye', 'up'
}

all_stopwords = sastrawi_stopwords.union(custom_stopwords)

def remove_stopwords(tokens):
    return [t for t in tokens if t not in all_stopwords and len(t) > 2]

df['step4_no_stopwords'] = df['step3_normalized'].apply(remove_stopwords)
df[['step3_normalized', 'step4_no_stopwords']].head(5)

## 3.5 Tahap 5: Stemming (Sastrawi dengan Kamus Cache)
Mengubah setiap kata berimbuhan menjadi kata dasar (*root word*) menggunakan algoritma Nazief & Adriani (PySastrawi). Menggunakan *dictionary cache* pada kata unik agar proses stemming ribuan dokumen berjalan sangat cepat.

In [ ]:
stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

# Kumpulkan seluruh kata unik agar proses stemming hanya dilakukan 1x per kata unik
unique_words = set(word for tokens in df['step4_no_stopwords'] for word in tokens)
print(f'Jumlah kata unik yang akan di-stem: {len(unique_words)} kata')

stem_cache = {word: stemmer.stem(word) for word in unique_words}

def stem_tokens(tokens):
    stemmed = [stem_cache.get(t, t) for t in tokens]
    # Filter ulang stopwords jika hasil stemming menghasilkan kata stopword atau panjang <= 2
    return [t for t in stemmed if t not in all_stopwords and len(t) > 2]

df['step5_stemmed_tokens'] = df['step4_no_stopwords'].apply(stem_tokens)
df['text_preprocessed'] = df['step5_stemmed_tokens'].apply(lambda tokens: ' '.join(tokens))

# Filter dokumen yang setelah preprocessing menyisakan minimal 3 kata bermakna
df_final = df[df['step5_stemmed_tokens'].apply(len) >= 3].reset_index(drop=True)
print(f'Jumlah dokumen akhir setelah seluruh tahapan Text Preprocessing: {len(df_final)}')
df_final[['text_cleaned', 'step5_stemmed_tokens', 'text_preprocessed']].head(10)

## 3.6 Menyimpan Hasil Text Preprocessing
File `03_preprocessed_data.csv` akan digunakan sebagai input pada notebook **`04_Text_Representation.ipynb`**.

In [ ]:
df_final.to_csv('03_preprocessed_data.csv', index=False)
print('Data hasil Text Preprocessing berhasil disimpan ke: 03_preprocessed_data.csv')